# Raw Data Exploration & Preprocessing Audit

Exploratory notebook for inspecting raw datasets (`01_raw`) and prototyping stateless cleaning functions before refactoring them into `src/album_recs/pipelines/data_processing/nodes.py` (`02_intermediate`).

* **`tracks_features.csv` (Spotify 1.2M+ Songs):** Previously explored and refactored into `preprocess_tracks` in `nodes.py` (standardizing join keys, removing missing/duplicate tracks, extracting `primary_artist`, and parsing `release_date` and `year`).

In [26]:
%load_ext kedro.ipython

The kedro.ipython extension is already loaded. To reload it, use:
  %reload_ext kedro.ipython


In [27]:
tracks = catalog.load("tracks_features")
tracks.head()

[11/16/26 13:43:20] INFO     Loading data from tracks_features (CSVDataset)...                 ]8;id=4000450;file://c:\projects\album-recs\album-recs311\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=4000451;file://c:\projects\album-recs\album-recs311\Lib\site-packages\kedro\io\data_catalog.py#1238\1238]8;;\

,id,name,album,album_id,artists,artist_ids,track_number,disc_number,explicit,danceability,...,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms,time_signature,year,release_date
0,7lmeHLHBe4nmXzuXc0HDjk,Testify,The Battle Of Los Angeles,2eia0myWFgoHuttJytCxgX,['Rage Against The Machine'],['2d0hyoQ5ynDBnkvAbJKORj'],1,1,False,0.470,...,0.0727,0.02610,0.000011,0.3560,0.503,117.906,210133,4,1999,02/11/1999
1,1wsRitfRRtWyEapl0q22o8,Guerrilla Radio,The Battle Of Los Angeles,2eia0myWFgoHuttJytCxgX,['Rage Against The Machine'],['2d0hyoQ5ynDBnkvAbJKORj'],2,1,True,0.599,...,0.1880,0.01290,0.000071,0.1550,0.489,103.680,206200,4,1999,02/11/1999
2,1hR0fIFK2qRG3f3RF70pb7,Calm Like a Bomb,The Battle Of Los Angeles,2eia0myWFgoHuttJytCxgX,['Rage Against The Machine'],['2d0hyoQ5ynDBnkvAbJKORj'],3,1,False,0.315,...,0.4830,0.02340,0.000002,0.1220,0.370,149.749,298893,4,1999,02/11/1999
3,2lbASgTSoDO7MTuLAXlTW0,Mic Check,The Battle Of Los Angeles,2eia0myWFgoHuttJytCxgX,['Rage Against The Machine'],['2d0hyoQ5ynDBnkvAbJKORj'],4,1,True,0.440,...,0.2370,0.16300,0.000004,0.1210,0.574,96.752,213640,4,1999,02/11/1999
4,1MQTmpYOZ6fcMQc56Hdo7T,Sleep Now In the Fire,The Battle Of Los Angeles,2eia0myWFgoHuttJytCxgX,['Rage Against The Machine'],['2d0hyoQ5ynDBnkvAbJKORj'],5,1,False,0.426,...,0.0701,0.00162,0.105000,0.0789,0.539,127.059,205600,4,1999,02/11/1999


In [28]:
tracks.dtypes


id                      str
name                    str
album                   str
album_id                str
artists                 str
artist_ids              str
track_number          int64
disc_number           int64
explicit               bool
danceability        float64
energy              float64
key                   int64
loudness            float64
mode                  int64
speechiness         float64
acousticness        float64
instrumentalness    float64
liveness            float64
valence             float64
tempo               float64
duration_ms           int64
time_signature        int64
year                  int64
release_date            str
dtype: object

In [ ]:
import ast
import pandas as pd

# Quick diagnostic across all 1.2M rows
print("Shape:", tracks.shape)
print("Nulls:\n", tracks.isnull().sum()[tracks.isnull().sum() > 0])
print("Duplicate track IDs:", tracks["id"].duplicated().sum())
print("Year range:", tracks["year"].min(), "to", tracks["year"].max())
print("Zero/invalid years:", (tracks["year"] <= 0).sum())

Shape: (1048575, 24)
Nulls:
 name      3
album    11
dtype: int64
Duplicate track IDs: 0
Year range: 0 to 2020
Zero/invalid years: 10


In [30]:
print("Any nulls:\n", tracks.isnull().sum()[tracks.isnull().sum() > 0])
display(tracks.describe().loc[["min", "max"]])

Any nulls:
 name      3
album    11
dtype: int64


,track_number,disc_number,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms,time_signature,year
min,1.0,1.0,0.0,0.0,0.0,-60.000,0.0,0.000,0.000,0.0,0.0,0.0,0.000,1000.0,0.0,0.0
max,50.0,13.0,1.0,1.0,11.0,7.234,1.0,0.969,0.996,1.0,1.0,1.0,248.934,6061090.0,5.0,2020.0


In [ ]:
def _parse_first_item(x: pd.Series) -> pd.Series:
    return x.apply(lambda val: ast.literal_eval(val)[0] if pd.notna(val) and val != "[]" else None)

In [ ]:
def _parse_release_date(x: pd.Series) -> pd.Series:
    return pd.to_datetime(x, format="mixed", dayfirst=True, errors="coerce")

In [ ]:
def _parse_year(x: pd.Series) -> pd.Series:
    year = pd.to_numeric(x, errors="coerce")
    year = year.where(year > 0, pd.NA) # 0 used as placeholder when year unknown
    return year.astype("Int64")

In [ ]:
def preprocess_tracks(df: pd.DataFrame) -> pd.DataFrame:

    df = df.rename(
        columns={
            "id":"track_id",
            "name":"track_name",
            "album":"album_name"        
        }
    )

    df = df.dropna(subset=["track_id", "album_id", "track_name", "album_name", "artists"]).drop_duplicates(subset=["track_id"])

    df["primary_artist"] = _parse_first_item(df["artists"])
    df["primary_artist_id"] = _parse_first_item(df["artist_ids"])
    df = df.drop(columns=["artists","artists_ids"])

    df["release_date"] = _parse_release_date(df["release_date"])
    df["year"] = _parse_year(df["year"])

    return df